In [ ]:
"""
Tableau de Bord — Suivi des Pépinières
Lancer : python dashboard_pepinieres.py
Ouvrir : http://127.0.0.1:8050
"""

import dash
from dash import dcc, html, dash_table, Input, Output, callback
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────────────────────
#  CHARGEMENT & NETTOYAGE
# ─────────────────────────────────────────────────────────────────────────────
df_raw = pd.read_excel("Tableau_de_bord_pepinieres__version_1_.xlsx")
df_raw["Semés"] = pd.to_numeric(df_raw["Semés"], errors="coerce")
df_raw["Taux_Germination"] = pd.to_numeric(df_raw["Taux_Germination"], errors="coerce")
df_raw["Taux_Mortalite"] = df_raw["Morts"] / df_raw["Germés"].replace(0, np.nan)

# Ordre trimestriel logique
TRIM_ORDER = ["T1 2025", "T2 2025", "T3 2025", "T4 2025"]

# Coordonnées GPS des pays
GPS = {
    "Burkina Faso": (12.36, -1.53),
    "Mali":         (17.57, -3.99),
    "Niger":        (17.60,  8.08),
    "Ghana":        ( 7.95, -1.02),
    "Sénégal":      (14.50,-14.45),
}

# ─────────────────────────────────────────────────────────────────────────────
#  PALETTE & STYLES
# ─────────────────────────────────────────────────────────────────────────────
C = {
    "bg":      "#0e1a14",
    "card":    "#162310",
    "border":  "#2a3d28",
    "a1":      "#4ade80",   # vert vif
    "a2":      "#86efac",   # vert clair
    "a3":      "#fbbf24",   # amber
    "a4":      "#f87171",   # rouge doux
    "a5":      "#60a5fa",   # bleu
    "a6":      "#c084fc",   # violet
    "text":    "#e2f0e6",
    "sub":     "#7aab82",
    "grid":    "#1f2f1e",
}

PIE_COLORS  = [C["a1"], C["a2"], C["a3"], C["a4"], C["a5"], C["a6"], "#34d399", "#fb923c"]
BASE_LAYOUT = dict(
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    font=dict(family="'Outfit', sans-serif", color=C["text"], size=12),
    margin=dict(l=10, r=10, t=40, b=10),
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(size=11)),
    xaxis=dict(gridcolor=C["grid"], zerolinecolor=C["grid"], tickfont=dict(size=11)),
    yaxis=dict(gridcolor=C["grid"], zerolinecolor=C["grid"], tickfont=dict(size=11)),
)

# ─────────────────────────────────────────────────────────────────────────────
#  COMPOSANTS UI
# ─────────────────────────────────────────────────────────────────────────────
def card(children, extra_style=None):
    style = {
        "background": C["card"],
        "border": f"1px solid {C['border']}",
        "borderRadius": "14px",
        "padding": "20px 22px",
    }
    if extra_style:
        style.update(extra_style)
    return html.Div(children, style=style)


def kpi_card(icon, label, value_id, delta_id, color):
    return html.Div([
        html.Div(icon, style={"fontSize": "1.6rem", "marginBottom": "8px"}),
        html.Div(label, style={"fontSize": ".72rem", "color": C["sub"],
                               "textTransform": "uppercase", "letterSpacing": ".08em"}),
        html.Div(id=value_id, style={"fontSize": "1.65rem", "fontWeight": "700",
                                     "letterSpacing": "-.02em", "margin": "4px 0",
                                     "color": C["text"]}),
        html.Div(id=delta_id, style={"fontSize": ".75rem", "color": color}),
    ], style={
        "background": C["card"],
        "border": f"1px solid {C['border']}",
        "borderRadius": "14px",
        "padding": "18px 20px",
        "position": "relative",
        "overflow": "hidden",
        "borderTop": f"3px solid {color}",
        "transition": "transform .2s",
    })


def filter_dropdown(label, fid, options, multi=True):
    return html.Div([
        html.Label(label, style={"fontSize": ".7rem", "color": C["sub"],
                                 "textTransform": "uppercase", "letterSpacing": ".07em",
                                 "marginBottom": "5px", "display": "block"}),
        dcc.Dropdown(
            id=fid,
            options=[{"label": o, "value": o} for o in options],
            multi=multi,
            placeholder=f"Tous ({len(options)})",
            style={"fontSize": "13px"},
            className="dark-drop",
        )
    ], style={"flex": "1", "minWidth": "160px"})


# ─────────────────────────────────────────────────────────────────────────────
#  APP
# ─────────────────────────────────────────────────────────────────────────────
app = dash.Dash(
    __name__,
    external_stylesheets=[
        "https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700&family=Space+Mono&display=swap"
    ],
    title="Tableau de Bord — Pépinières",
)

app.index_string = """<!DOCTYPE html>
<html>
<head>{%metas%}<title>{%title%}</title>{%favicon%}{%css%}
<style>
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
body{background:#0e1a14;font-family:'Outfit',sans-serif;color:#e2f0e6;min-height:100vh}
::-webkit-scrollbar{width:5px}::-webkit-scrollbar-track{background:#0e1a14}
::-webkit-scrollbar-thumb{background:#2a3d28;border-radius:3px}

/* Dropdown dark theme */
.dark-drop .Select-control{background:#1a2e18!important;border-color:#2a3d28!important;color:#e2f0e6!important;border-radius:8px!important}
.dark-drop .Select-menu-outer{background:#1a2e18!important;border-color:#2a3d28!important;z-index:9999}
.dark-drop .Select-option{background:#1a2e18!important;color:#e2f0e6!important}
.dark-drop .Select-option:hover,.dark-drop .Select-option.is-focused{background:#2a3d28!important}
.dark-drop .Select-value-label{color:#e2f0e6!important}
.dark-drop .Select-placeholder{color:#7aab82!important}
.dark-drop .Select--multi .Select-value{background:#2a3d28!important;border-color:#4ade80!important;color:#e2f0e6!important;border-radius:6px!important}
.dark-drop input{color:#e2f0e6!important}
.dash-dropdown .Select-arrow-zone .Select-arrow{border-top-color:#7aab82!important}

/* Select2-style */
.Select-control,.Select-menu-outer{font-family:'Outfit',sans-serif!important}

/* Reset button */
.reset-btn{background:none;border:1px solid #2a3d28;color:#7aab82;padding:6px 14px;border-radius:8px;cursor:pointer;font-family:'Outfit',sans-serif;font-size:.75rem;letter-spacing:.06em;text-transform:uppercase;transition:all .2s}
.reset-btn:hover{border-color:#4ade80;color:#4ade80}
</style>
</head>
<body>{%app_entry%}<footer>{%config%}{%scripts%}{%renderer%}</footer>
</body></html>"""

# ── Layout ────────────────────────────────────────────────────────────────────
app.layout = html.Div([

    # ── En-tête ──────────────────────────────────────────────────────────────
    html.Div([
        html.Div([
            html.Div(style={"width": "12px", "height": "12px", "borderRadius": "50%",
                            "background": "linear-gradient(135deg,#4ade80,#86efac)",
                            "boxShadow": "0 0 14px rgba(74,222,128,.5)"}),
            html.Div([
                html.Span("🌱 Tableau de Bord — Pépinières", style={
                    "fontSize": "1.15rem", "fontWeight": "700",
                    "background": "linear-gradient(90deg,#e2f0e6,#7aab82)",
                    "-webkit-background-clip": "text",
                    "-webkit-text-fill-color": "transparent",
                }),
                html.Span("  Suivi de la production & mortalité 2025",
                          style={"fontSize": ".78rem", "color": C["sub"], "marginLeft": "10px"}),
            ]),
        ], style={"display": "flex", "alignItems": "center", "gap": "12px"}),
        html.Span("2025  |  5 pays  |  69 enregistrements",
                  style={"fontSize": ".75rem", "color": C["sub"],
                         "fontFamily": "'Space Mono',monospace"}),
    ], style={
        "background": "linear-gradient(135deg,#162310,#0e1a14)",
        "borderBottom": f"1px solid {C['border']}",
        "padding": "18px 32px",
        "display": "flex", "alignItems": "center", "justifyContent": "space-between",
    }),

    # ── Corps principal ───────────────────────────────────────────────────────
    html.Div([

        # ── Filtres ───────────────────────────────────────────────────────────
        card([
            html.Div([
                html.Div("🔍", style={"fontSize": "1.1rem", "marginRight": "8px"}),
                html.Span("Filtres", style={"fontWeight": "600", "fontSize": ".85rem",
                                            "marginRight": "auto"}),
                html.Button("Réinitialiser", id="btn-reset", n_clicks=0, className="reset-btn"),
            ], style={"display": "flex", "alignItems": "center", "marginBottom": "14px"}),

            html.Div([
                filter_dropdown("Pays", "f-pays",
                    sorted(df_raw["Pays"].dropna().unique())),
                filter_dropdown("Région", "f-region",
                    sorted(df_raw["Région"].dropna().unique())),
                filter_dropdown("Type d'organisation", "f-type",
                    sorted(df_raw["Type_org"].dropna().unique())),
                filter_dropdown("Trimestre", "f-trim",
                    [t for t in TRIM_ORDER if t in df_raw["Trimestre"].unique()]),
            ], style={"display": "flex", "gap": "16px", "flexWrap": "wrap"}),
        ], {"marginBottom": "20px"}),

        # ── Cartes KPI ────────────────────────────────────────────────────────
        html.Div([
            kpi_card("🌱", "Total Semés",       "kpi-semes-val",    "kpi-semes-sub",    C["a1"]),
            kpi_card("🌿", "Total Germés",      "kpi-germes-val",   "kpi-germes-sub",   C["a2"]),
            kpi_card("🚚", "Total Distribués",  "kpi-distrib-val",  "kpi-distrib-sub",  C["a5"]),
            kpi_card("💀", "Total Morts",       "kpi-morts-val",    "kpi-morts-sub",    C["a4"]),
            kpi_card("📈", "Taux Germination",  "kpi-txgerm-val",   "kpi-txgerm-sub",   C["a3"]),
            kpi_card("📉", "Taux Mortalité",    "kpi-txmort-val",   "kpi-txmort-sub",   C["a4"]),
            kpi_card("🏭", "Capacité Totale",   "kpi-capa-val",     "kpi-capa-sub",     C["a6"]),
            kpi_card("🪴", "Pots Préparés",     "kpi-pots-val",     "kpi-pots-sub",     C["a3"]),
        ], style={"display": "grid",
                  "gridTemplateColumns": "repeat(4,1fr)",
                  "gap": "14px", "marginBottom": "20px"}),

        # ── Ligne 1 : Histogramme pays + Carte géo ────────────────────────────
        html.Div([
            card([dcc.Graph(id="fig-histo-pays",
                            config={"displayModeBar": False},
                            style={"height": "340px"})]),
            card([dcc.Graph(id="fig-carte",
                            config={"displayModeBar": False},
                            style={"height": "340px"})]),
        ], style={"display": "grid", "gridTemplateColumns": "1.4fr 1fr",
                  "gap": "16px", "marginBottom": "16px"}),

        # ── Ligne 2 : Camembert type org + Camembert espèces ─────────────────
        html.Div([
            card([dcc.Graph(id="fig-pie-org",
                            config={"displayModeBar": False},
                            style={"height": "320px"})]),
            card([dcc.Graph(id="fig-pie-espece",
                            config={"displayModeBar": False},
                            style={"height": "320px"})]),
        ], style={"display": "grid", "gridTemplateColumns": "1fr 1fr",
                  "gap": "16px", "marginBottom": "16px"}),

        # ── Ligne 3 : Courbe trimestre + Barres mortalité/problèmes ──────────
        html.Div([
            card([dcc.Graph(id="fig-courbe-trim",
                            config={"displayModeBar": False},
                            style={"height": "320px"})]),
            card([dcc.Graph(id="fig-causes",
                            config={"displayModeBar": False},
                            style={"height": "320px"})]),
        ], style={"display": "grid", "gridTemplateColumns": "1.2fr 1fr",
                  "gap": "16px", "marginBottom": "16px"}),

        # ── Ligne 4 : Histogramme région + Problèmes détaillés ────────────────
        html.Div([
            card([dcc.Graph(id="fig-histo-region",
                            config={"displayModeBar": False},
                            style={"height": "300px"})]),
            card([dcc.Graph(id="fig-problemes",
                            config={"displayModeBar": False},
                            style={"height": "300px"})]),
        ], style={"display": "grid", "gridTemplateColumns": "1fr 1fr",
                  "gap": "16px", "marginBottom": "16px"}),

        # ── Tableau détail ────────────────────────────────────────────────────
        card([
            html.Div("📋  Données détaillées", style={
                "fontWeight": "600", "fontSize": ".9rem",
                "marginBottom": "14px", "color": C["text"],
            }),
            dash_table.DataTable(
                id="table-detail",
                columns=[
                    {"name": c, "id": c} for c in
                    ["Pays","Région","Organisation","Type_org","Espèce",
                     "Trimestre","Semés","Germés","Distribués","Morts",
                     "Taux_Germination","Cause_mortalité"]
                ],
                sort_action="native",
                filter_action="native",
                page_size=12,
                page_action="native",
                style_table={"overflowX": "auto"},
                style_header={
                    "backgroundColor": "#0e1a14", "color": C["sub"],
                    "fontWeight": "600", "fontSize": "11px",
                    "textTransform": "uppercase", "letterSpacing": ".06em",
                    "border": "none", "borderBottom": f"1px solid {C['border']}",
                    "padding": "10px 14px",
                },
                style_cell={
                    "backgroundColor": C["card"], "color": C["text"],
                    "fontFamily": "'Outfit',sans-serif", "fontSize": "12.5px",
                    "border": "none", "borderBottom": f"1px solid {C['grid']}",
                    "padding": "9px 14px", "textAlign": "left",
                    "whiteSpace": "normal", "maxWidth": "160px",
                },
                style_data_conditional=[
                    {"if": {"row_index": "odd"}, "backgroundColor": "#131f10"},
                    {"if": {"filter_query": "{Taux_Germination} > 0.85"},
                     "color": C["a1"], "fontWeight": "600"},
                    {"if": {"filter_query": "{Morts} > 500"},
                     "color": C["a4"]},
                ],
            ),
        ], {"marginBottom": "24px"}),

    ], style={"padding": "24px 32px", "maxWidth": "1500px", "margin": "0 auto"}),

])  # fin layout


# ─────────────────────────────────────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def apply_filters(pays, region, type_org, trim):
    d = df_raw.copy()
    if pays:   d = d[d["Pays"].isin(pays)]
    if region: d = d[d["Région"].isin(region)]
    if type_org: d = d[d["Type_org"].isin(type_org)]
    if trim:   d = d[d["Trimestre"].isin(trim)]
    return d


def fmt_k(n):
    if n >= 1_000_000: return f"{n/1_000_000:.1f}M"
    if n >= 1_000:     return f"{n/1_000:.1f}k"
    return str(int(n))


# ─────────────────────────────────────────────────────────────────────────────
#  RESET FILTRES
# ─────────────────────────────────────────────────────────────────────────────
@app.callback(
    Output("f-pays",   "value"),
    Output("f-region", "value"),
    Output("f-type",   "value"),
    Output("f-trim",   "value"),
    Input("btn-reset", "n_clicks"),
    prevent_initial_call=True,
)
def reset_filters(_):
    return None, None, None, None


# ─────────────────────────────────────────────────────────────────────────────
#  CALLBACK PRINCIPAL
# ─────────────────────────────────────────────────────────────────────────────
@app.callback(
    # KPIs
    Output("kpi-semes-val",  "children"), Output("kpi-semes-sub",  "children"),
    Output("kpi-germes-val", "children"), Output("kpi-germes-sub", "children"),
    Output("kpi-distrib-val","children"), Output("kpi-distrib-sub","children"),
    Output("kpi-morts-val",  "children"), Output("kpi-morts-sub",  "children"),
    Output("kpi-txgerm-val", "children"), Output("kpi-txgerm-sub", "children"),
    Output("kpi-txmort-val", "children"), Output("kpi-txmort-sub", "children"),
    Output("kpi-capa-val",   "children"), Output("kpi-capa-sub",   "children"),
    Output("kpi-pots-val",   "children"), Output("kpi-pots-sub",   "children"),
    # Graphiques
    Output("fig-histo-pays",   "figure"),
    Output("fig-carte",        "figure"),
    Output("fig-pie-org",      "figure"),
    Output("fig-pie-espece",   "figure"),
    Output("fig-courbe-trim",  "figure"),
    Output("fig-causes",       "figure"),
    Output("fig-histo-region", "figure"),
    Output("fig-problemes",    "figure"),
    # Tableau
    Output("table-detail", "data"),
    # Inputs
    Input("f-pays",   "value"),
    Input("f-region", "value"),
    Input("f-type",   "value"),
    Input("f-trim",   "value"),
)
def update_all(pays, region, type_org, trim):
    d = apply_filters(pays, region, type_org, trim)
    n = len(d)

    # ── KPIs ─────────────────────────────────────────────────────────────────
    total_semes   = pd.to_numeric(d["Semés"],    errors="coerce").sum()
    total_germes  = d["Germés"].sum()
    total_distrib = d["Distribués"].sum()
    total_morts   = d["Morts"].sum()
    tx_germ       = d["Taux_Germination"].mean()
    tx_mort       = (d["Morts"] / d["Germés"].replace(0, np.nan)).mean()
    capa_tot      = d["Capacité"].sum()
    pots_tot      = d["Pots_préparés"].sum()

    kpis_out = [
        fmt_k(total_semes),  f"{n} enregistrements filtrés",
        fmt_k(total_germes), f"{tx_germ*100:.1f}% taux germination moyen",
        fmt_k(total_distrib),f"{total_distrib/max(total_germes,1)*100:.1f}% des germés",
        fmt_k(total_morts),  f"{total_morts/max(total_germes,1)*100:.1f}% des germés",
        f"{tx_germ*100:.1f}%", "taux moyen sur la sélection",
        f"{tx_mort*100:.1f}%" if not np.isnan(tx_mort) else "—",
                              "mortalité / germés",
        fmt_k(capa_tot),     f"{n} pépinières",
        fmt_k(pots_tot),     f"{pots_tot/max(capa_tot,1)*100:.0f}% de la capacité",
    ]

    # ── 1. Histogramme pays ───────────────────────────────────────────────────
    dp = d.groupby("Pays")[["Semés","Germés","Distribués","Morts"]].sum().reset_index()
    dp["Semés"] = pd.to_numeric(dp["Semés"], errors="coerce")
    fig_pays = go.Figure()
    for col, color, name in [
        ("Semés",     C["a1"], "Semés"),
        ("Germés",    C["a2"], "Germés"),
        ("Distribués",C["a5"], "Distribués"),
        ("Morts",     C["a4"], "Morts"),
    ]:
        fig_pays.add_trace(go.Bar(
            x=dp["Pays"], y=dp[col], name=name,
            marker_color=color, marker_line_width=0,
        ))
    fig_pays.update_layout(**BASE_LAYOUT,
        title="Production par pays", barmode="group",
        bargap=0.18, bargroupgap=0.06)

    # ── 2. Carte géographique ─────────────────────────────────────────────────
    map_d = d.groupby("Pays").agg(
        Semés=("Semés", lambda x: pd.to_numeric(x, errors="coerce").sum()),
        Germés=("Germés","sum"),
        Morts=("Morts","sum"),
    ).reset_index()
    map_d["lat"] = map_d["Pays"].map(lambda p: GPS.get(p, (0,0))[0])
    map_d["lon"] = map_d["Pays"].map(lambda p: GPS.get(p, (0,0))[1])
    map_d["taille"] = np.sqrt(map_d["Semés"].clip(lower=0)) * 0.8
    map_d["texte"]  = map_d.apply(
        lambda r: f"<b>{r['Pays']}</b><br>Semés : {fmt_k(r['Semés'])}"
                  f"<br>Germés : {fmt_k(r['Germés'])}<br>Morts : {fmt_k(r['Morts'])}", axis=1)

    fig_carte = go.Figure(go.Scattergeo(
        lat=map_d["lat"], lon=map_d["lon"],
        text=map_d["texte"], hoverinfo="text",
        mode="markers+text",
        textposition="top center",
        textfont=dict(color=C["text"], size=11),
        marker=dict(
            size=map_d["taille"].clip(lower=8, upper=50),
            color=map_d["Germés"], colorscale="Greens",
            showscale=True, line_width=1, line_color=C["border"],
            colorbar=dict(title="Germés", thickness=10,
                          tickfont=dict(color=C["sub"], size=9)),
        ),
    ))
    fig_carte.update_geos(
        scope="africa",
        bgcolor="rgba(0,0,0,0)",
        landcolor="#1a2e18", oceancolor="#0e1a14",
        coastlinecolor=C["border"], countrycolor=C["border"],
        showland=True, showocean=True, showcountries=True,
        showcoastlines=True,
        center=dict(lat=14, lon=-2), projection_scale=3.5,
    )
    fig_carte.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        font=dict(family="'Outfit',sans-serif", color=C["text"]),
        margin=dict(l=0, r=0, t=36, b=0),
        title="Carte géographique — Production",
    )

    # ── 3. Camembert type organisation ────────────────────────────────────────
    do = d.groupby("Type_org")["Germés"].sum().reset_index()
    fig_pie_org = go.Figure(go.Pie(
        labels=do["Type_org"], values=do["Germés"], hole=0.52,
        marker=dict(colors=PIE_COLORS, line=dict(color=C["bg"], width=2)),
        textfont=dict(color=C["text"]),
    ))
    fig_pie_org.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        font=dict(family="'Outfit',sans-serif", color=C["text"]),
        margin=dict(l=10, r=10, t=40, b=10),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
        title="Germés par type d'organisation",
    )

    # ── 4. Camembert espèces ──────────────────────────────────────────────────
    de = d.groupby("Espèce")["Semés"].apply(
        lambda x: pd.to_numeric(x, errors="coerce").sum()).reset_index()
    fig_pie_esp = go.Figure(go.Pie(
        labels=de["Espèce"], values=de["Semés"], hole=0.52,
        marker=dict(colors=PIE_COLORS[::-1], line=dict(color=C["bg"], width=2)),
        textfont=dict(color=C["text"]),
    ))
    fig_pie_esp.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        font=dict(family="'Outfit',sans-serif", color=C["text"]),
        margin=dict(l=10, r=10, t=40, b=10),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
        title="Semés par espèce",
    )

    # ── 5. Courbe évolution trimestrielle ─────────────────────────────────────
    dt = d.groupby("Trimestre")[["Semés","Germés","Distribués","Morts"]].sum().reset_index()
    dt["Semés"] = pd.to_numeric(dt["Semés"], errors="coerce")
    dt = dt.set_index("Trimestre").reindex(
        [t for t in TRIM_ORDER if t in dt["Trimestre"].values if t in TRIM_ORDER]
    ).reset_index()
    # fallback si réindex plante
    if dt.empty or dt["Trimestre"].isna().all():
        dt = d.groupby("Trimestre")[["Semés","Germés","Distribués","Morts"]].sum().reset_index()
        dt["Semés"] = pd.to_numeric(dt["Semés"], errors="coerce")

    fig_trim = go.Figure()
    for col, color, name in [
        ("Semés",     C["a1"], "Semés"),
        ("Germés",    C["a2"], "Germés"),
        ("Distribués",C["a5"], "Distribués"),
        ("Morts",     C["a4"], "Morts"),
    ]:
        fig_trim.add_trace(go.Scatter(
            x=dt["Trimestre"], y=dt[col],
            name=name, mode="lines+markers",
            line=dict(color=color, width=2.5),
            marker=dict(size=8, color=color),
            fill="tozeroy" if col == "Semés" else "none",
            fillcolor="rgba(74,222,128,0.05)" if col == "Semés" else None,
        ))
    fig_trim.update_layout(**BASE_LAYOUT, title="Évolution par trimestre")

    # ── 6. Barres causes de mortalité ─────────────────────────────────────────
    dc = d["Cause_mortalité"].value_counts().reset_index()
    dc.columns = ["Cause", "Nb"]
    fig_causes = go.Figure(go.Bar(
        y=dc["Cause"], x=dc["Nb"], orientation="h",
        marker=dict(color=PIE_COLORS[:len(dc)], line_width=0),
        text=dc["Nb"], textposition="outside",
        textfont=dict(color=C["text"]),
    ))
    fig_causes.update_layout(**BASE_LAYOUT,
        title="Causes de mortalité",
        xaxis_title="Nombre d'occurrences",
        yaxis=dict(gridcolor=C["grid"], zerolinecolor=C["grid"]),
        margin=dict(l=10, r=40, t=40, b=10),
    )

    # ── 7. Histogramme région ─────────────────────────────────────────────────
    dr = d.groupby("Région")[["Semés","Germés","Distribués"]].sum().reset_index()
    dr["Semés"] = pd.to_numeric(dr["Semés"], errors="coerce")
    fig_region = go.Figure()
    for col, color, name in [
        ("Semés",     C["a1"], "Semés"),
        ("Germés",    C["a2"], "Germés"),
        ("Distribués",C["a5"], "Distribués"),
    ]:
        fig_region.add_trace(go.Bar(
            x=dr["Région"], y=dr[col], name=name,
            marker_color=color, marker_line_width=0,
        ))
    fig_region.update_layout(**BASE_LAYOUT,
        title="Production par région", barmode="group",
        bargap=0.2, bargroupgap=0.08)

    # ── 8. Barres problèmes détaillés ────────────────────────────────────────
    prob = (d["Problèmes"].dropna()
              .str.split(", ").explode().str.strip()
              .value_counts().reset_index())
    prob.columns = ["Problème", "Nb"]
    fig_prob = go.Figure(go.Bar(
        y=prob["Problème"], x=prob["Nb"], orientation="h",
        marker=dict(
            color=prob["Nb"],
            colorscale=[[0, C["a2"]], [0.5, C["a3"]], [1, C["a4"]]],
            line_width=0,
        ),
        text=prob["Nb"], textposition="outside",
        textfont=dict(color=C["text"]),
    ))
    fig_prob.update_layout(**BASE_LAYOUT,
        title="Problèmes signalés",
        margin=dict(l=10, r=40, t=40, b=10),
        xaxis_title="Occurrences",
        yaxis=dict(gridcolor=C["grid"], zerolinecolor=C["grid"]),
    )

    # ── Tableau ───────────────────────────────────────────────────────────────
    table_cols = ["Pays","Région","Organisation","Type_org","Espèce",
                  "Trimestre","Semés","Germés","Distribués","Morts",
                  "Taux_Germination","Cause_mortalité"]
    tbl = d[table_cols].copy()
    tbl["Taux_Germination"] = tbl["Taux_Germination"].apply(
        lambda x: f"{x*100:.1f}%" if pd.notna(x) else "—")
    tbl["Semés"] = pd.to_numeric(tbl["Semés"], errors="coerce").astype("Int64")

    return (
        *kpis_out,
        fig_pays, fig_carte,
        fig_pie_org, fig_pie_esp,
        fig_trim, fig_causes,
        fig_region, fig_prob,
        tbl.to_dict("records"),
    )


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("\n✅  Dashboard Pépinières → http://127.0.0.1:8050\n")
    app.run(debug=True, port=8050)

FileNotFoundError: [Errno 2] No such file or directory: 'Tableau_de_bord_pepinieres__version_1_.xlsx'

In [3]:
import pandas as pd

In [9]:
import os

In [10]:
df_raw = pd.read_excel("C:\Users\ASUS\Dash_Pepin_26\Tableau_de_bord_pepinieres__version_1_.xlsx")

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (3414532384.py, line 1)

In [7]:
# Fix 1: Use raw string by adding 'r' prefix
df_raw = pd.read_excel(r"C:\Users\ASUS\Dash_Pepin_26\Tableau_de_bord_pepinieres__version_1_.xlsx")

# OR Fix 2: Use forward slashes instead of backslashes
# df_raw = pd.read_excel("C:/Users/ASUS/Dash_Pepin_26/Tableau_de_bord_pepinieres__version_1_.xlsx")

# OR Fix 3: Use double backslashes to escape them
# df_raw = pd.read_excel("C:\\Users\\ASUS\\Dash_Pepin_26\\Tableau_de_bord_pepinieres__version_1_.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ASUS\\Dash_Pepin_26\\Tableau_de_bord_pepinieres__version_1_.xlsx'

In [8]:
# Option 1: Verify the exact path and filename
# Use the raw string prefix 'r' to avoid escape character issues
df_raw = pd.read_excel(r"C:\Users\ASUS\Dash_Pepin_26\Tableau_de_bord_pepinieres__version_1_.xlsx")

# Option 2: If you're unsure about the exact location, try using a relative path
# This assumes the file is in the same directory as your notebook
# df_raw = pd.read_excel("Tableau_de_bord_pepinieres__version_1_.xlsx")

# Option 3: If you need to check what files are available in a directory
import os
# Print all files in the directory to check if your file exists
# print(os.listdir(r"C:\Users\ASUS\Dash_Pepin_26"))

# Option 4: Use a file dialog to select the file interactively
# from tkinter import filedialog
# file_path = filedialog.askopenfilename(title="Select Excel file", 
#                                       filetypes=[("Excel files", "*.xlsx")])
# df_raw = pd.read_excel(file_path)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ASUS\\Dash_Pepin_26\\Tableau_de_bord_pepinieres__version_1_.xlsx'